# 🔢 Python Sorting Algorithms — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Think of sorting like organizing a hand of playing cards. You have several strategies: scan and swap neighbors (bubble), pick the smallest and move it to the front (selection), pick up each card and slide it into position (insertion), split the deck in half and merge two ordered halves (merge), pin a pivot card and partition around it (quick), or build a heap and pull the max off the top repeatedly (heap). Every strategy is a trade-off between comparisons, swaps, memory, and stability.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [Visual Model — Array State Diagrams](#1) |
| 2 | [Complexity at a Glance](#2) |
| 3 | [Decision Map — Which Sort When](#3) |
| 4 | [Pattern 1: Bubble Sort](#4) |
| 5 | [Pattern 2: Insertion Sort](#5) |
| 6 | [Pattern 3: Merge Sort](#6) |
| 7 | [Pattern 4: Quick Sort — Lomuto & Hoare](#7) |
| 8 | [Pattern 5: Heap Sort](#8) |
| 9 | [Stability Demo](#9) |
| 10 | [Pattern 6: Counting Sort](#10) |
| 11 | [Pattern 7: Radix Sort](#11) |
| 12 | [Pattern 8: Bucket Sort](#12) |
| 13 | [LC 912 — Sort an Array (from scratch)](#13) |
| 14 | [LC 148 — Sort List (merge sort on linked list)](#14) |
| 15 | [Full Decision Map](#15) |
| 16 | [Interview Cheat Sheet](#16) |


<a id='1'></a>

## 1. Visual Model — Array State Diagrams

```
BUBBLE SORT — swap neighbors until no swaps needed
pass 1:  [5, 3, 8, 1]  → compare 5,3 → swap  → [3, 5, 8, 1]
         [3, 5, 8, 1]  → compare 5,8 → ok    → [3, 5, 8, 1]
         [3, 5, 8, 1]  → compare 8,1 → swap  → [3, 5, 1, 8]  ← 8 bubbled to end
pass 2:  [3, 5, 1, 8]  → compare 3,5 → ok
         [3, 5, 1, 8]  → compare 5,1 → swap  → [3, 1, 5, 8]
pass 3:  [3, 1, 5, 8]  → compare 3,1 → swap  → [1, 3, 5, 8]  ✓ done

INSERTION SORT — pick next card, slide left into its slot
start:   [5, 3, 8, 1]
key=3:   [5, _, 8, 1] → shift 5 right → [_, 5, 8, 1] → insert 3 → [3, 5, 8, 1]
key=8:   8 >= 5, no shift needed                               → [3, 5, 8, 1]
key=1:   shift 8,5,3 right                                     → [1, 3, 5, 8]  ✓

MERGE SORT — split in half, sort halves, merge
[5, 3, 8, 1]
  ├── [5, 3]           split
  │     ├── [5]        base case
  │     └── [3]        base case
  │     └── merge → [3, 5]
  └── [8, 1]           split
        ├── [8]
        └── [1]
        └── merge → [1, 8]
  └── merge [3,5] + [1,8] → [1, 3, 5, 8]  ✓

QUICK SORT (Lomuto) — pivot = last element, partition around it
[5, 3, 8, 1]   pivot=1
  i=-1, scan j=0..2:
    j=0: 5>1 skip
    j=1: 3>1 skip
    j=2: 8>1 skip
  swap arr[i+1] with pivot → [1, 3, 8, 5]   pivot lands at index 0
  recurse left=[]  right=[3,8,5]  ...

STABILITY: stable sort preserves relative order of equal keys
  data = [(3,'a'), (1,'b'), (3,'c')]
  stable:   → [(1,'b'), (3,'a'), (3,'c')]   'a' still before 'c'
  unstable: → [(1,'b'), (3,'c'), (3,'a')]   order of 3s could flip
```


<a id='2'></a>

## 2. Complexity at a Glance

```
ALGORITHM        BEST       AVERAGE    WORST      SPACE   STABLE   IN-PLACE
────────────────────────────────────────────────────────────────────────────
Bubble Sort      O(n)       O(n²)      O(n²)      O(1)    YES      YES
Insertion Sort   O(n)       O(n²)      O(n²)      O(1)    YES      YES
Selection Sort   O(n²)      O(n²)      O(n²)      O(1)    NO       YES
Merge Sort       O(n log n) O(n log n) O(n log n) O(n)    YES      NO
Quick Sort       O(n log n) O(n log n) O(n²)      O(log n) NO      YES
Heap Sort        O(n log n) O(n log n) O(n log n) O(1)    NO       YES
Timsort (Python) O(n)       O(n log n) O(n log n) O(n)    YES      NO
────────────────────────────────────────────────────────────────────────────
Counting Sort    O(n+k)     O(n+k)     O(n+k)     O(k)    YES      NO
Radix Sort       O(nk)      O(nk)      O(nk)      O(n+k)  YES      NO
Bucket Sort      O(n+k)     O(n+k)     O(n²)      O(n)    YES*     NO
────────────────────────────────────────────────────────────────────────────
k = range of values   * stable if insertion sort used within buckets
```


<a id='3'></a>

## 3. Decision Map — Which Sort When

```
SIGNAL IN THE PROBLEM                     USE THIS
────────────────────────────────────────────────────────────────────────
"sort in-place, limited memory"           Quick Sort or Heap Sort
"stable sort required"                    Merge Sort or Timsort
"nearly-sorted data"                      Insertion Sort  O(n) best case
"sort small array (< 20 elements)"        Insertion Sort  low constant
"integers in known range [0..k]"          Counting Sort
"integers, sort digit by digit"           Radix Sort
"floats uniformly in [0, 1)"              Bucket Sort
"sort linked list"                        Merge Sort  (no random access)
"k-th largest element"                    Quickselect  (partial quick sort)
"external sort (data > memory)"           Merge Sort  (sequential access)
"interview: implement from scratch"       Merge Sort or Quick Sort
"Python built-in"                         list.sort() / sorted()  (Timsort)
```


<a id='4'></a>

## 4. 🧩 Pattern 1: Bubble Sort — LC 912

---

```
PROBLEM:  Sort an array in ascending order — implement bubble sort from scratch.

APPROACH: Repeatedly scan left-to-right, swapping adjacent out-of-order pairs.
          After each pass, the largest unsorted element "bubbles" to its final position.
          Optimization: if no swaps in a pass, the array is already sorted — stop early.

SLOW MOTION TRACE on [5, 3, 8, 1]:
  pass 1  j=0: 5>3 swap  → [3,5,8,1]
          j=1: 5<8 ok    → [3,5,8,1]
          j=2: 8>1 swap  → [3,5,1,8]   8 is in place
  pass 2  j=0: 3<5 ok
          j=1: 5>1 swap  → [3,1,5,8]   5 is in place
  pass 3  j=0: 3>1 swap  → [1,3,5,8]   done
  early-exit: pass 4 would find 0 swaps → break

KEY INSIGHT: After pass i, the last i elements are in their final positions.
             The early-exit optimization makes it O(n) on already-sorted input.

TIME / SPACE:
  Best:    O(n)   — already sorted, 1 pass with 0 swaps
  Average: O(n²)  — n*(n-1)/2 comparisons
  Worst:   O(n²)  — reverse sorted
  Space:   O(1)   — in-place, no extra memory
```


In [ ]:
from typing import List

def bubble_sort(nums: List[int]) -> List[int]:
    """
    LC 912 — Sort an Array (Bubble Sort implementation).
    Approach: O(n²) comparison sort — swap neighbors until sorted.
    Args:
        nums (List[int]): unsorted list of integers.
    Returns:
        List[int]: sorted list in ascending order (in-place, returns same list).
    Time:  O(n²) average/worst — O(n) best (already sorted, early exit fires)
    Space: O(1) — no auxiliary array, swaps in-place
    """
    n = len(nums)
    for i in range(n):                        # after pass i, last i elements are sorted
        swapped = False                        # early-exit sentinel
        for j in range(n - 1 - i):            # don't compare already-sorted tail
            if nums[j] > nums[j + 1]:         # out of order — the bigger one must move right
                nums[j], nums[j + 1] = nums[j + 1], nums[j]   # swap
                swapped = True
        if not swapped:                        # entire pass had no swaps — already sorted
            break
    return nums

# Slow motion on nums = [5, 3, 8, 1]:
# pass 0: j=0 5>3 swap->[3,5,8,1]  j=1 5<8 ok  j=2 8>1 swap->[3,5,1,8]  swapped=True
# pass 1: j=0 3<5 ok  j=1 5>1 swap->[3,1,5,8]  swapped=True
# pass 2: j=0 3>1 swap->[1,3,5,8]  swapped=True
# pass 3: j=0 1<3 ok — swapped=False → break

def test_harness(fn):
    tests = [
        ([5, 3, 8, 1], [1, 3, 5, 8]),
        ([1], [1]),
        ([], []),
        ([2, 2, 2], [2, 2, 2]),
        ([5, 4, 3, 2, 1], [1, 2, 3, 4, 5]),
        ([1, 2, 3, 4, 5], [1, 2, 3, 4, 5]),     # already sorted — early exit fires
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(list(inputs[0]))                # copy so in-place doesn't corrupt test
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(bubble_sort)
print("bubble_sort defined.")

# Simplicity and clarity is Gold


<a id='5'></a>

## 5. 🧩 Pattern 2: Insertion Sort — Best for Nearly-Sorted

---

```
PROBLEM:  Sort in-place. Best algorithm when input is nearly sorted (online streams,
          small arrays, or the inner loop of Timsort).

APPROACH: Maintain a sorted left portion. For each new element, slide it left
          past every element larger than it until it reaches its correct slot.

SLOW MOTION TRACE on [3, 5, 1, 4, 2]:
  i=1  key=5: 5>=3 no shift needed          → [3, 5, 1, 4, 2]
  i=2  key=1: shift 5→right, shift 3→right  → [1, 3, 5, 4, 2]
  i=3  key=4: shift 5→right                 → [1, 3, 4, 5, 2]
  i=4  key=2: shift 5,4,3→right             → [1, 2, 3, 4, 5]

KEY INSIGHT: On nearly-sorted data each key travels only 1–2 slots left.
             That's why Timsort uses insertion sort for runs shorter than ~64.

TIME / SPACE:
  Best:    O(n)   — already sorted, inner loop never runs
  Average: O(n²)  — each element moves ~n/4 positions
  Worst:   O(n²)  — reverse sorted
  Space:   O(1)   — in-place
```


In [ ]:
from typing import List

def insertion_sort(nums: List[int]) -> List[int]:
    """
    Insertion Sort — best for nearly-sorted data, stable, in-place.
    Approach: walk right; slide each key left into its sorted position.
    Args:
        nums (List[int]): unsorted list.
    Returns:
        List[int]: sorted list (in-place).
    Time:  O(n²) worst/average — O(n) best on already-sorted
    Space: O(1) — no auxiliary memory
    """
    for i in range(1, len(nums)):
        key = nums[i]                  # the card we're inserting into sorted left hand
        j = i - 1
        while j >= 0 and nums[j] > key:   # slide bigger elements right to make room
            nums[j + 1] = nums[j]
            j -= 1
        nums[j + 1] = key              # drop key into the gap we made
    return nums

# Slow motion on [3, 5, 1]:
# i=1 key=5: nums[0]=3 < 5 → stop. nums[1]=5. [3,5,1]
# i=2 key=1: nums[1]=5>1 shift→[3,5,5,1]  nums[0]=3>1 shift→[3,3,5,1]  j=-1 stop
#            nums[0]=1 → [1,3,5]

def test_harness(fn):
    tests = [
        ([3, 5, 1, 4, 2], [1, 2, 3, 4, 5]),
        ([1], [1]),
        ([], []),
        ([5, 5, 5], [5, 5, 5]),
        ([2, 1], [1, 2]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(list(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(insertion_sort)
print("insertion_sort defined.")

# Simplicity and clarity is Gold


<a id='6'></a>

## 6. 🧩 Pattern 3: Merge Sort — LC 148, 912

---

```
PROBLEM:  Sort stably in O(n log n). Required when: data is a linked list
          (no random access), external sort, or stability is mandatory.

APPROACH: Divide array in half recursively until size 1 (base case).
          Merge two sorted halves into a new sorted array.
          Key: merging two sorted arrays takes O(n) — just walk two pointers.

SLOW MOTION TRACE on [5, 3, 8, 1]:
  split: [5,3] | [8,1]
  split: [5] | [3]     [8] | [1]
  merge [5]+[3]:   5>3  → take 3, take 5  → [3,5]
  merge [8]+[1]:   8>1  → take 1, take 8  → [1,8]
  merge [3,5]+[1,8]:
    compare 3 vs 1  → take 1  → [1]
    compare 3 vs 8  → take 3  → [1,3]
    compare 5 vs 8  → take 5  → [1,3,5]
    8 remaining     → take 8  → [1,3,5,8] ✓

KEY INSIGHT: "When would you pick merge over quick?"
  - Linked lists: merge sort only needs sequential access; quick sort needs pivot swap
  - Stability required: merge is stable, quick is not
  - Worst-case guarantee: merge is always O(n log n); quick degrades to O(n²)
  - External sort: merge works file-to-file; quick requires random access

TIME / SPACE:
  All cases: O(n log n) — log n levels, O(n) work per level
  Space:     O(n) — temporary arrays during merge (cannot be done truly in-place)
```


In [ ]:
from typing import List

def merge_sort(nums: List[int]) -> List[int]:
    """
    Merge Sort — stable, O(n log n) guaranteed, divide & conquer.
    Approach: split in half recursively, merge two sorted halves.
    Args:
        nums (List[int]): unsorted list.
    Returns:
        List[int]: new sorted list (not in-place — returns new array).
    Time:  O(n log n) — log n levels of recursion, O(n) merge per level
    Space: O(n) — O(n) for merge temp arrays + O(log n) call stack
    """
    if len(nums) <= 1:
        return nums                    # base case: array of 0 or 1 is already sorted

    mid = len(nums) // 2
    left  = merge_sort(nums[:mid])     # sort left half
    right = merge_sort(nums[mid:])     # sort right half
    return _merge(left, right)         # combine two sorted halves

def _merge(left: List[int], right: List[int]) -> List[int]:
    """Merge two sorted arrays into one sorted array in O(n) time."""
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:        # <= preserves stability (left wins ties)
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])            # drain whichever half had leftovers
    result.extend(right[j:])
    return result

# Slow motion merge of [3,5] and [1,8]:
# i=0 j=0: left[0]=3 > right[0]=1  → take right[0]=1  j=1
# i=0 j=1: left[0]=3 < right[1]=8  → take left[0]=3   i=1
# i=1 j=1: left[1]=5 < right[1]=8  → take left[1]=5   i=2
# i=2: left exhausted → extend right[1:] = [8]
# result: [1,3,5,8]

def test_harness(fn):
    tests = [
        ([5, 3, 8, 1], [1, 3, 5, 8]),
        ([1], [1]),
        ([], []),
        ([3, 3, 1, 2], [1, 2, 3, 3]),
        ([5, 4, 3, 2, 1], [1, 2, 3, 4, 5]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(list(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(merge_sort)
print("merge_sort defined.")

# Simplicity and clarity is Gold


<a id='7'></a>

## 7. 🧩 Pattern 4: Quick Sort — Lomuto & Hoare Partitions

---

```
PROBLEM:  Sort in-place with O(n log n) average. Best practical sort for
          random data. LC 215 (Quickselect) uses the same partition logic.

LOMUTO PARTITION — pivot = last element
  [3, 6, 8, 10, 1, 2, 1]   pivot=1
  i=-1   scan j left to right:
    j=0: 3>1  skip
    j=1: 6>1  skip
    j=2: 8>1  skip
    j=3: 10>1 skip
    j=4: 1<=1  i++ → swap arr[0] with arr[4]  → [1, 6, 8, 10, 3, 2, 1]
    j=5: 2>1  skip
  swap arr[i+1]=arr[1] with pivot arr[6]=1  → [1, 1, 8, 10, 3, 2, 6]
  pivot at index 1. Left=[1], Right=[8,10,3,2,6]

HOARE PARTITION — pivot = first element, two pointers moving inward
  Faster in practice — 3x fewer swaps than Lomuto on average.
  [3, 6, 8, 10, 1, 2]  pivot=3
  lo=0 hi=5
    lo++ until arr[lo]>=pivot: lo=2 (8>=3)
    hi-- until arr[hi]<=pivot: hi=5 (2<=3)... wait hi=4 (1<=3)
    lo<=hi: swap arr[2] with arr[4] → [3,6,1,10,8,2]
    lo=3, hi=3... continue

KEY INSIGHT: Lomuto is simpler to code; Hoare is more efficient.
             Worst case (sorted input) is O(n²) — avoid with random pivot or median-of-3.

TIME / SPACE:
  Average: O(n log n) — random pivot splits evenly on average
  Worst:   O(n²) — pivot always min or max (sorted/reverse-sorted input)
  Space:   O(log n) — call stack depth (in-place)
```


In [ ]:
from typing import List
import random

def quick_sort(nums: List[int]) -> List[int]:
    """
    Quick Sort — in-place, O(n log n) average, unstable.
    Uses Lomuto partition with random pivot to avoid O(n²) worst case.
    Args:
        nums (List[int]): unsorted list (modified in-place).
    Returns:
        List[int]: same list, sorted.
    Time:  O(n log n) average — random pivot prevents degenerate splits
    Space: O(log n) — call stack (balanced), O(n) worst case (unbalanced)
    """
    _quick_sort(nums, 0, len(nums) - 1)
    return nums

def _quick_sort(nums, lo, hi):
    if lo < hi:
        pivot_idx = _lomuto_partition(nums, lo, hi)
        _quick_sort(nums, lo, pivot_idx - 1)    # sort left of pivot
        _quick_sort(nums, pivot_idx + 1, hi)    # sort right of pivot

def _lomuto_partition(nums, lo, hi):
    """Lomuto partition: pivot=last element, i tracks boundary of <=pivot region."""
    # Random pivot: swap random element to end to prevent O(n²) on sorted input
    rand_idx = random.randint(lo, hi)
    nums[rand_idx], nums[hi] = nums[hi], nums[rand_idx]

    pivot = nums[hi]                   # pivot is now the last element
    i = lo - 1                         # i = right edge of "<=pivot" region (starts empty)
    for j in range(lo, hi):
        if nums[j] <= pivot:           # this element belongs in the left partition
            i += 1
            nums[i], nums[j] = nums[j], nums[i]   # expand left partition
    nums[i + 1], nums[hi] = nums[hi], nums[i + 1]  # place pivot between partitions
    return i + 1                       # pivot's final index

# Lomuto trace on [3, 1, 2] pivot=2 (last):
# i=-1  j=0: 3>2 skip
#        j=1: 1<=2 i=0 swap arr[0] with arr[1] → [1,3,2]
# swap arr[i+1]=arr[1] with arr[hi]=arr[2] → [1,2,3]   pivot at index 1 ✓

# ── Hoare partition (more efficient — shown for reference) ────────────────────
def _hoare_partition(nums, lo, hi):
    """Hoare: two inward-moving pointers, ~3x fewer swaps than Lomuto."""
    pivot = nums[lo]
    i = lo - 1
    j = hi + 1
    while True:
        i += 1
        while nums[i] < pivot:         # find element >= pivot from left
            i += 1
        j -= 1
        while nums[j] > pivot:         # find element <= pivot from right
            j -= 1
        if i >= j:
            return j                   # partition index (not pivot's final position)
        nums[i], nums[j] = nums[j], nums[i]

def test_harness(fn):
    tests = [
        ([5, 3, 8, 1], [1, 3, 5, 8]),
        ([1], [1]),
        ([], []),
        ([3, 3, 1, 2], [1, 2, 3, 3]),
        ([5, 4, 3, 2, 1], [1, 2, 3, 4, 5]),    # reverse sorted — random pivot handles it
        ([1, 2, 3, 4, 5], [1, 2, 3, 4, 5]),    # already sorted
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(list(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(quick_sort)
print("quick_sort defined (Lomuto + Hoare shown).")

# Simplicity and clarity is Gold


<a id='8'></a>

## 8. 🧩 Pattern 5: Heap Sort — in-place, O(n log n) guaranteed

---

```
PROBLEM:  Sort in-place with O(n log n) guaranteed (no worst-case O(n²) like quick).

APPROACH: Phase 1 — build a max-heap from the array (heapify, O(n)).
          Phase 2 — repeatedly extract the max: swap root with last, shrink heap, sift-down.

SLOW MOTION TRACE on [4, 1, 3, 2, 5]:
  Phase 1 — build max-heap:
    start from last non-leaf = index n//2 - 1 = 1
    sift-down index 1: arr[1]=1, children=2,5. 5>1 → swap → [4,5,3,2,1]
    sift-down index 0: arr[0]=4, children=5,3. 5>4 → swap → [5,4,3,2,1]
    max-heap: [5, 4, 3, 2, 1]   root=5 is max ✓

  Phase 2 — extract max repeatedly:
    swap root 5 with last 1 → [1,4,3,2, | 5]   heap size=4
    sift-down index 0: 1 < 4 → swap → [4,1,3,2, | 5]
                               1 < 2 → swap → [4,2,3,1, | 5]
    swap root 4 with last 2 → [2,2,3,1 | 4,5]... (continue)
    result: [1,2,3,4,5] ✓

KEY INSIGHT: Heap sort uses the heap property rather than comparisons between
             arbitrary pairs — sift-down always moves the max to the root in O(log n).

TIME / SPACE:
  All cases: O(n log n) — O(n) heapify + n * O(log n) extractions
  Space:     O(1) — truly in-place, no extra array
```


In [ ]:
from typing import List

def heap_sort(nums: List[int]) -> List[int]:
    """
    Heap Sort — in-place, O(n log n) guaranteed, unstable.
    Approach: build max-heap, then repeatedly extract max to sorted tail.
    Args:
        nums (List[int]): unsorted list (modified in-place).
    Returns:
        List[int]: same list, sorted ascending.
    Time:  O(n log n) — O(n) build-heap + n * O(log n) sift-downs
    Space: O(1) — no extra memory, in-place
    """
    n = len(nums)

    # Phase 1: build max-heap — start from last non-leaf, sift each down
    for i in range(n // 2 - 1, -1, -1):   # last non-leaf = n//2-1
        _sift_down(nums, i, n)

    # Phase 2: extract max repeatedly — swap root with end, shrink heap
    for end in range(n - 1, 0, -1):
        nums[0], nums[end] = nums[end], nums[0]   # max lands in sorted tail
        _sift_down(nums, 0, end)                   # restore heap property in [0..end)

    return nums

def _sift_down(nums, root, heap_size):
    """Push nums[root] down to its correct position in max-heap of size heap_size."""
    while True:
        largest = root
        left    = 2 * root + 1       # left child index
        right   = 2 * root + 2       # right child index

        if left < heap_size and nums[left] > nums[largest]:
            largest = left           # left child is bigger
        if right < heap_size and nums[right] > nums[largest]:
            largest = right          # right child is even bigger

        if largest == root:
            break                    # root is already the largest — heap property holds

        nums[root], nums[largest] = nums[largest], nums[root]   # swap down
        root = largest               # follow the swap path down

# Slow motion sift-down on [1,4,3,2] heap_size=4, root=0:
# largest=0  left=1: 4>1 → largest=1  right=2: 3<4 → largest stays 1
# swap arr[0] with arr[1] → [4,1,3,2]  root=1
# left=3: 2>1 → largest=3  right=4: out of range
# swap arr[1] with arr[3] → [4,2,3,1]  root=3
# left=7: out of range → break

def test_harness(fn):
    tests = [
        ([4, 1, 3, 2, 5], [1, 2, 3, 4, 5]),
        ([1], [1]),
        ([], []),
        ([3, 3, 1], [1, 3, 3]),
        ([5, 4, 3, 2, 1], [1, 2, 3, 4, 5]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(list(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(heap_sort)
print("heap_sort defined.")

# Simplicity and clarity is Gold


<a id='9'></a>

## 9. Stability Demo — Why It Matters

```
STABILITY: a sort is stable if equal elements keep their original relative order.

EXAMPLE: Sort employees by salary. Then sort by department.
  If the second sort is stable, employees within each department remain salary-ordered.
  If unstable, salary order within each department is scrambled.

  data = [(3,'alice'), (1,'bob'), (3,'charlie'), (2,'dave')]
         ─── key ───   ──── value ────
  Stable sort by key:   [(1,'bob'), (2,'dave'), (3,'alice'), (3,'charlie')]
                                                          ↑ alice before charlie (original order)
  Unstable sort result: [(1,'bob'), (2,'dave'), (3,'charlie'), (3,'alice')]
                                                          ↑ could flip

WHICH ARE STABLE?
  ✅ Bubble, Insertion, Merge, Timsort, Counting, Radix, Bucket (with insertion)
  ❌ Selection, Quick, Heap

INTERVIEW ANSWER: "I'd use merge sort here because stability is required and
  we're sorting a linked list — merge sort uses only sequential access."
```


In [ ]:
# ── Stability demonstration ────────────────────────────────────────────────────
# Show that stable sort preserves relative order of equal keys

data = [(3, 'alice'), (1, 'bob'), (3, 'charlie'), (2, 'dave')]
# Original order: alice is at index 0, charlie at index 2

# Python's sorted() is Timsort — stable
result_stable = sorted(data, key=lambda x: x[0])
print("Stable sort result:")
for item in result_stable:
    print(f"  {item}")
# (1,'bob') (2,'dave') (3,'alice') (3,'charlie') — alice before charlie ✓

print()

# Simulate unstable: manually reverse the 3-group to show the difference
result_unstable = sorted(data, key=lambda x: x[0])
# flip the 3s to demonstrate unstable behavior
threes = [(k,v) for k,v in result_unstable if k == 3]
threes_flipped = list(reversed(threes))
result_simulated_unstable = [(k,v) for k,v in result_unstable if k != 3] + threes_flipped
result_simulated_unstable.sort(key=lambda x: x[0])  # re-sort to interleave correctly
# Just show the concept manually:
print("If unstable — 3-group could be:")
print(f"  (3,'charlie') before (3,'alice')  — relative order flipped")

print()

# Multi-key sort: sort by department then by salary — stable preserves salary order
employees = [
    ('Engineering', 90000, 'Alice'),
    ('Marketing',   70000, 'Bob'),
    ('Engineering', 80000, 'Charlie'),
    ('Marketing',   75000, 'Dave'),
    ('Engineering', 85000, 'Eve'),
]

# Step 1: sort by salary
by_salary = sorted(employees, key=lambda e: e[1])
print("Sorted by salary:")
for e in by_salary:
    print(f"  {e}")

# Step 2: stable sort by department — salary order within each dept is preserved
by_dept_then_salary = sorted(by_salary, key=lambda e: e[0])
print("\nThen stable sort by department (salary order preserved within dept):")
for e in by_dept_then_salary:
    print(f"  {e}")

# Simplicity and clarity is Gold


<a id='10'></a>

## 10. 🧩 Pattern 6: Counting Sort — LC 75, 274

---

```
PROBLEM:  Sort integers in a known range [0..k] in O(n+k) — faster than O(n log n)
          when k is small relative to n.

APPROACH: Count how many times each value appears. Compute prefix sums to find
          output positions. Place each element in its correct output position.

SLOW MOTION TRACE on [2, 0, 2, 1, 1, 0]  (k=2):
  Step 1 — count:   counts = [2, 2, 2]   (0 appears 2x, 1 appears 2x, 2 appears 2x)
  Step 2 — prefix:  prefix = [2, 4, 6]   (prefix[i] = number of elements <= i)
  Step 3 — place (scan original right-to-left for stability):
    val=0: pos=prefix[0]-1=1  output[1]=0  prefix[0]=1
    val=1: pos=prefix[1]-1=3  output[3]=1  prefix[1]=3
    val=1: pos=prefix[1]-1=2  output[2]=1  prefix[1]=2
    val=2: pos=prefix[2]-1=5  output[5]=2  prefix[2]=5
    ...
    output = [0, 0, 1, 1, 2, 2] ✓

CONSTRAINT: Only works on integers (or mappable keys). k must be reasonable.
            LC 75 (Sort Colors 0,1,2) — k=2, classic counting sort.

TIME / SPACE:
  Time:  O(n+k) — n to count, k to prefix-sum, n to place
  Space: O(k)   — counts array of size k+1
```


In [ ]:
from typing import List

def counting_sort(nums: List[int]) -> List[int]:
    """
    Counting Sort — O(n+k) for integers in range [min..max].
    Approach: count occurrences, prefix-sum for positions, place in output.
    Args:
        nums (List[int]): list of integers (can have negatives).
    Returns:
        List[int]: new sorted list.
    Time:  O(n + k) where k = max - min + 1
    Space: O(n + k) — counts array + output array
    """
    if not nums:
        return []
    lo, hi = min(nums), max(nums)
    k = hi - lo + 1
    counts = [0] * k

    for val in nums:
        counts[val - lo] += 1          # shift by lo so negatives work

    # Expand counts back into sorted output
    output = []
    for i, cnt in enumerate(counts):
        output.extend([i + lo] * cnt)  # repeat each value by its count
    return output

# LC 75 — Sort Colors: 0s, 1s, 2s — k=3, counting sort is exact fit
def sort_colors(nums: List[int]) -> None:
    """
    LC 75 — Sort Colors (Dutch National Flag / Counting Sort).
    Approach: count 0s, 1s, 2s then overwrite — O(n) time, O(1) space.
    Args:
        nums (List[int]): list containing only 0, 1, 2. Modified in-place.
    Returns:
        None (in-place).
    Time:  O(n) — one pass to count, one pass to overwrite
    Space: O(1) — only 3 counters
    """
    c0 = c1 = c2 = 0
    for x in nums:
        if x == 0: c0 += 1
        elif x == 1: c1 += 1
        else: c2 += 1
    # Overwrite in-place: first c0 zeros, then c1 ones, then c2 twos
    nums[:c0]           = [0] * c0
    nums[c0:c0+c1]      = [1] * c1
    nums[c0+c1:]        = [2] * c2

# Slow motion on [2,0,2,1,1,0]:
# counts = [2,2,2]  (index 0→2 ones, 1→2 ones, 2→2 ones)
# output: 0,0 then 1,1 then 2,2 → [0,0,1,1,2,2] ✓

def test_harness_counting(fn):
    tests = [
        ([2, 0, 2, 1, 1, 0], [0, 0, 1, 1, 2, 2]),
        ([1], [1]),
        ([], []),
        ([3, 1, 2], [1, 2, 3]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(list(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness_counting(counting_sort)
print("counting_sort defined.")

# Test sort_colors
colors = [2, 0, 2, 1, 1, 0]
sort_colors(colors)
print(f"sort_colors result: {colors}")    # [0,0,1,1,2,2]
print("sort_colors defined.")

# Simplicity and clarity is Gold


<a id='11'></a>

## 11. 🧩 Pattern 7: Radix Sort — LSD digit-by-digit

---

```
PROBLEM:  Sort integers without comparing them — O(nk) where k = number of digits.
          Beats O(n log n) when k is small (e.g., 32-bit ints → k=10 base-10 digits).

APPROACH: LSD (Least Significant Digit) — sort by digit position, right-to-left.
          Use a stable sort (counting sort) at each digit position.
          After k passes, the array is sorted.

SLOW MOTION TRACE on [170, 45, 75, 90, 802, 24, 2, 66]:
  Pass 1 (ones digit):
    170→0  45→5  75→5  90→0  802→2  24→4  2→2  66→6
    buckets: 0:[170,90]  2:[802,2]  4:[24]  5:[45,75]  6:[66]
    after:  [170, 90, 802, 2, 24, 45, 75, 66]

  Pass 2 (tens digit):
    170→7  90→9  802→0  2→0  24→2  45→4  75→7  66→6
    buckets: 0:[802,2]  2:[24]  4:[45]  6:[66]  7:[170,75]  9:[90]
    after:  [802, 2, 24, 45, 66, 170, 75, 90]

  Pass 3 (hundreds digit):
    802→8  2→0  24→0  45→0  66→0  170→1  75→0  90→0
    buckets: 0:[2,24,45,66,75,90]  1:[170]  8:[802]
    after:  [2, 24, 45, 66, 75, 90, 170, 802] ✓

KEY INSIGHT: Stability at each pass is critical — earlier passes establish
             sub-ordering that later passes must preserve.

TIME / SPACE:
  Time:  O(nk)  — k passes, each O(n+10) ≈ O(n)
  Space: O(n+10) — one set of 10 buckets + output array
```


In [ ]:
from typing import List

def radix_sort(nums: List[int]) -> List[int]:
    """
    Radix Sort (LSD) — O(nk) for non-negative integers, k = digit count.
    Approach: stable counting sort per digit position, right-to-left.
    Args:
        nums (List[int]): non-negative integers.
    Returns:
        List[int]: new sorted list.
    Time:  O(n * k) where k = digits in max value (≤ 10 for 32-bit ints)
    Space: O(n + 10) — 10 buckets (base 10) + output array
    """
    if not nums:
        return []

    max_val = max(nums)
    exp = 1                            # current digit position: 1=ones, 10=tens, etc.

    output = list(nums)                # working copy
    while max_val // exp > 0:
        output = _counting_sort_by_digit(output, exp)
        exp *= 10

    return output

def _counting_sort_by_digit(nums, exp):
    """Stable sort by the (exp)s digit — 10 buckets for base-10."""
    n = len(nums)
    output = [0] * n
    count  = [0] * 10                  # digits 0-9

    for val in nums:
        digit = (val // exp) % 10      # extract the digit at this position
        count[digit] += 1

    for i in range(1, 10):            # prefix sum: count[i] = output position after digit i
        count[i] += count[i - 1]

    for i in range(n - 1, -1, -1):   # scan right-to-left for stability
        digit = (nums[i] // exp) % 10
        count[digit] -= 1
        output[count[digit]] = nums[i]

    return output

# Slow motion trace — ones digit of [170, 45, 75]:
# digit(170)=0  digit(45)=5  digit(75)=5
# count = [1,0,0,0,0,2,0,0,0,0]
# prefix = [1,1,1,1,1,3,3,3,3,3]
# scan right-to-left:
#   75 → digit=5 → pos=prefix[5]-1=2 → output[2]=75  count[5]=2
#   45 → digit=5 → pos=prefix[5]-1=1 → output[1]=45  count[5]=1
#   170 → digit=0 → pos=prefix[0]-1=0 → output[0]=170  count[0]=0
# output = [170, 45, 75]  ← stable: 45 before 75 ✓

def test_harness(fn):
    tests = [
        ([170, 45, 75, 90, 802, 24, 2, 66], [2, 24, 45, 66, 75, 90, 170, 802]),
        ([1], [1]),
        ([], []),
        ([0, 0, 0], [0, 0, 0]),
        ([100, 10, 1], [1, 10, 100]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(list(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(radix_sort)
print("radix_sort defined.")

# Simplicity and clarity is Gold


<a id='12'></a>

## 12. 🧩 Pattern 8: Bucket Sort — floats in [0, 1)

---

```
PROBLEM:  Sort n floats uniformly distributed in [0, 1) in expected O(n).
          Works for any distribution with a known range — split into n buckets.

APPROACH: Create n empty buckets. Map each float to bucket index = floor(val * n).
          Sort each bucket (insertion sort — each bucket has ~O(1) elements on avg).
          Concatenate sorted buckets.

SLOW MOTION TRACE on [0.78, 0.17, 0.39, 0.26, 0.72, 0.94, 0.21, 0.12]:
  n=8 buckets (index 0-7)
  val → bucket: floor(val * 8)
    0.78 → 6    0.17 → 1    0.39 → 3    0.26 → 2
    0.72 → 5    0.94 → 7    0.21 → 1    0.12 → 0

  buckets:
    0: [0.12]
    1: [0.17, 0.21]   ← two in same bucket
    2: [0.26]
    3: [0.39]
    5: [0.72]
    6: [0.78]
    7: [0.94]

  sort each bucket (already sorted here by luck)
  concat: [0.12, 0.17, 0.21, 0.26, 0.39, 0.72, 0.78, 0.94] ✓

KEY INSIGHT: If data is uniform, each bucket gets ~1 element on average.
             Worst case is O(n²) if all elements land in one bucket.

TIME / SPACE:
  Average: O(n)   — uniform distribution, n buckets, ~1 element each
  Worst:   O(n²)  — all in one bucket (insertion sort on full array)
  Space:   O(n)   — n buckets
```


In [ ]:
from typing import List

def bucket_sort(nums: List[float]) -> List[float]:
    """
    Bucket Sort — expected O(n) for floats uniformly distributed in [0, 1).
    Approach: scatter into n buckets by floor(val*n), sort each, gather.
    Args:
        nums (List[float]): floats in [0.0, 1.0).
    Returns:
        List[float]: new sorted list.
    Time:  O(n) average — uniform distribution; O(n²) worst (all same bucket)
    Space: O(n) — n buckets each holding on average 1 element
    """
    if not nums:
        return []
    n = len(nums)
    buckets = [[] for _ in range(n)]      # n empty buckets

    for val in nums:
        idx = int(val * n)                 # map [0,1) → [0,n)
        idx = min(idx, n - 1)             # guard: val=1.0 would give idx=n
        buckets[idx].append(val)           # scatter into bucket

    for bucket in buckets:
        bucket.sort()                      # insertion sort implicit — small buckets

    return [val for bucket in buckets for val in bucket]   # gather — concat all

# Slow motion on [0.78, 0.17] with n=2:
# 0.78 → bucket int(0.78*2)=1
# 0.17 → bucket int(0.17*2)=0
# buckets = [[0.17], [0.78]]
# concat → [0.17, 0.78] ✓

def test_harness(fn):
    import math
    tests = [
        ([0.78, 0.17, 0.39, 0.26, 0.72], [0.17, 0.26, 0.39, 0.72, 0.78]),
        ([0.5], [0.5]),
        ([], []),
        ([0.1, 0.9, 0.5, 0.3, 0.7], [0.1, 0.3, 0.5, 0.7, 0.9]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(list(inputs[0]))
        close = all(math.isclose(a, b) for a, b in zip(got, expected))
        status = "PASSED" if close and len(got) == len(expected) else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (status == "PASSED")
    print(f"{passed}/{len(tests)} tests passed")

test_harness(bucket_sort)
print("bucket_sort defined.")

# Simplicity and clarity is Gold


<a id='13'></a>

## 13. 🧩 Pattern 9: LC 912 — Sort an Array (from scratch, no built-in)

---

```
PROBLEM:  Given an integer array nums, sort it in ascending order.
          You must implement the sorting algorithm from scratch — no sort() or sorted().

TRICK:    Use merge sort — O(n log n) guaranteed, stable, easiest to implement correctly.
          For interview: "I'll use merge sort — it's O(n log n) guaranteed,
          handles any input, and is stable."

SLOW MOTION TRACE on [5, 2, 3, 1]:
  merge_sort([5,2,3,1])
    left  = merge_sort([5,2]) = merge_sort([5]) + merge_sort([2]) → merge([5],[2]) = [2,5]
    right = merge_sort([3,1]) = merge_sort([3]) + merge_sort([1]) → merge([3],[1]) = [1,3]
    merge([2,5], [1,3]):
      2>1 → take 1, 2<3 → take 2, 5>3 → take 3, take 5 → [1,2,3,5] ✓

KEY INSIGHT: This is the same merge_sort from Pattern 3 — LC 912 is the
             canonical "implement sort from scratch" problem.

TIME / SPACE: O(n log n) / O(n)
```


In [ ]:
from typing import List

def sortArray(nums: List[int]) -> List[int]:
    """
    LC 912 — Sort an Array.
    Approach: merge sort — divide, sort halves, merge. No built-in sort used.
    Args:
        nums (List[int]): unsorted integers, -5*10^4 <= nums[i] <= 5*10^4.
    Returns:
        List[int]: sorted ascending.
    Time:  O(n log n) — log n levels, O(n) merge per level
    Space: O(n) — temp arrays during merge + O(log n) call stack
    """
    if len(nums) <= 1:
        return nums

    mid   = len(nums) // 2
    left  = sortArray(nums[:mid])
    right = sortArray(nums[mid:])

    # Merge two sorted halves
    merged = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i]); i += 1
        else:
            merged.append(right[j]); j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged

def test_harness(fn):
    tests = [
        ([5, 2, 3, 1], [1, 2, 3, 5]),
        ([5, 1, 1, 2, 0, 0], [0, 0, 1, 1, 2, 5]),
        ([1], [1]),
        ([-3, -1, -2], [-3, -2, -1]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(list(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(sortArray)
print("sortArray defined.")

# Simplicity and clarity is Gold


<a id='14'></a>

## 14. 🧩 Pattern 10: LC 148 — Sort List (merge sort on linked list)

---

```
PROBLEM:  Sort a singly linked list in O(n log n) time and O(1) space (excluding recursion).

TRICK:    Merge sort — the only comparison sort that works on linked lists.
          Quick sort needs pivot swapping (requires random access).
          Merge sort only needs sequential access: find-midpoint + merge.

  FIND MIDPOINT: slow/fast pointer (Floyd's)
    slow moves 1 step, fast moves 2 steps
    when fast reaches end, slow is at midpoint
    [1→4→3→2→5→2→NULL]
    slow=1,fast=1 → slow=4,fast=3 → slow=3,fast=2 → slow=2,fast=NULL
    midpoint = 2 (index 2) → split: [1,4,3] + [2,5,2]

  MERGE two sorted lists:
    Use a dummy head node — pointer manipulation, no new nodes needed.

KEY INSIGHT: O(1) space (beyond recursion stack) because we rewire the .next
             pointers of existing nodes — no new list is allocated.

TIME / SPACE:
  Time:  O(n log n) — log n splits, O(n) merge per level
  Space: O(log n)   — recursion stack depth
```


In [ ]:
from typing import Optional

class ListNode:
    def __init__(self, val=0, nxt=None):
        self.val = val
        self.next = nxt
    def __repr__(self):
        vals = []
        cur = self
        while cur:
            vals.append(str(cur.val))
            cur = cur.next
        return "->".join(vals)

def sortList(head: Optional[ListNode]) -> Optional[ListNode]:
    """
    LC 148 — Sort List.
    Approach: merge sort on linked list — find mid, split, sort halves, merge.
    Args:
        head (Optional[ListNode]): head of unsorted linked list.
    Returns:
        Optional[ListNode]: head of sorted linked list.
    Time:  O(n log n) — log n splits, O(n) merge per level
    Space: O(log n)   — recursion stack; nodes are rewired, not copied
    """
    if not head or not head.next:
        return head                    # base case: 0 or 1 node is already sorted

    # Step 1: find midpoint using slow/fast pointers
    slow, fast = head, head.next      # fast starts one ahead so slow stops at left-mid
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    mid = slow.next                    # mid is start of right half
    slow.next = None                   # sever the link — split into two lists

    # Step 2: sort each half
    left  = sortList(head)
    right = sortList(mid)

    # Step 3: merge two sorted lists
    dummy = ListNode(0)                # sentinel head — avoids edge case on first node
    cur   = dummy
    while left and right:
        if left.val <= right.val:      # <= for stability
            cur.next = left
            left = left.next
        else:
            cur.next = right
            right = right.next
        cur = cur.next
    cur.next = left if left else right # attach remaining tail
    return dummy.next

# Slow motion find-midpoint on [4->2->1->3]:
# slow=4,fast=2 start
# iter 1: slow=2, fast=3 (fast.next=3, fast.next.next=None)
# loop ends (fast.next=None), slow=2
# mid = slow.next = 1  →  split: [4->2] and [1->3]

def build_list(vals):
    dummy = ListNode(0)
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def list_to_arr(head):
    result = []
    while head:
        result.append(head.val)
        head = head.next
    return result

def test_harness(fn):
    tests = [
        ([4, 2, 1, 3], [1, 2, 3, 4]),
        ([-1, 5, 3, 4, 0], [-1, 0, 3, 4, 5]),
        ([], []),
        ([1], [1]),
    ]
    passed = 0
    for *inputs, expected in tests:
        lst = build_list(inputs[0])
        got = list_to_arr(fn(lst))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(sortList)
print("sortList defined.")

# Simplicity and clarity is Gold


<a id='15'></a>

## 15. Full Decision Map

```
QUESTION TYPE                          KEY TECHNIQUE          LC PROBLEMS
──────────────────────────────────────────────────────────────────────────────────
Sort array from scratch (interview)    Merge Sort             912
Sort linked list                       Merge Sort             148
Custom comparator / string concat key  cmp_to_key + sort      179
Integer range [0..k], k small          Counting Sort          75, 274
Multi-digit integers, digit-wise sort  Radix Sort             (912 variant)
Floats uniformly in [0,1)              Bucket Sort            164
K-th largest element                   Quickselect            215
Nearly-sorted data                     Insertion Sort         —
In-place, worst-case O(n log n)        Heap Sort              —
Stable sort required                   Merge Sort / Timsort   —
Python built-in                        list.sort()            All
Sort intervals by start                sorted(key=lambda)     56, 435
Sort by frequency                      Counter.most_common    347
──────────────────────────────────────────────────────────────────────────────────
```


<a id='16'></a>

## 16. Interview Cheat Sheet

### 1. When to reach for each sort:

| Signal | What to Do |
|--------|------------|
| "sort in O(n log n) guaranteed" | Merge or Heap sort |
| "sort linked list" | Merge sort only |
| "stability required" | Merge sort or Python sorted() |
| "integers, small range" | Counting sort |
| "k-th largest" | Quickselect (partition without full sort) |
| "nearly sorted" | Insertion sort |
| "external/disk sort" | Merge sort |

### 2. Copy-paste templates:

```python
# MERGE SORT — stable, O(n log n), any input
def merge_sort(nums):
    if len(nums) <= 1: return nums
    mid = len(nums) // 2
    left, right = merge_sort(nums[:mid]), merge_sort(nums[mid:])
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: out.append(left[i]); i += 1
        else: out.append(right[j]); j += 1
    return out + left[i:] + right[j:]

# COUNTING SORT — O(n+k), integers only
def counting_sort(nums):
    if not nums: return []
    lo, hi = min(nums), max(nums)
    counts = [0] * (hi - lo + 1)
    for v in nums: counts[v - lo] += 1
    return [i + lo for i, c in enumerate(counts) for _ in range(c)]

# QUICKSELECT — O(n) average for k-th largest
import random
def find_kth_largest(nums, k):
    pivot = random.choice(nums)
    gt = [x for x in nums if x > pivot]
    eq = [x for x in nums if x == pivot]
    lt = [x for x in nums if x < pivot]
    if k <= len(gt): return find_kth_largest(gt, k)
    if k <= len(gt) + len(eq): return pivot
    return find_kth_largest(lt, k - len(gt) - len(eq))
```

### 3. Gotchas to not forget:

```
❌  Quick sort is O(n²) on sorted input without random pivot
❌  Counting sort breaks on floats or large k
❌  Radix sort requires non-negative integers
❌  Bucket sort worst-case is O(n²) if data isn't uniform
✅  Python's sort() is Timsort — stable, O(n log n), use it in interviews unless told otherwise
✅  Merge sort is the answer for linked list sort — always
✅  Stability = equal keys keep original relative order
✅  Quickselect finds k-th in O(n) average — don't sort to find kth largest
```


```
SORTING ALGORITHMS MASTER MAP
═══════════════════════════════════════════════════════

                    SORT AN ARRAY
                         │
         ┌───────────────┼────────────────┐
         ▼               ▼                ▼
   COMPARISON      DISTRIBUTION      SPECIAL CASE
   SORTS           SORTS             SORTS
         │               │                │
   ┌─────┴──────┐  ┌─────┴──────┐   ┌────┴────────┐
   │O(n²) slow  │  │  Counting  │   │  Timsort    │
   │  Bubble    │  │  O(n+k)    │   │  Python     │
   │  Selection │  │  integers  │   │  built-in   │
   │  Insertion │  │  only      │   │  stable     │
   │  (good for │  ├────────────┤   └─────────────┘
   │  ~sorted)  │  │  Radix     │
   └────────────┘  │  O(nk)     │
                   │  digit     │
   ┌────────────┐  │  by digit  │
   │O(n log n)  │  ├────────────┤
   │  Merge ✅  │  │  Bucket    │
   │  stable    │  │  O(n) avg  │
   │  linked    │  │  floats    │
   │  list      │  │  in [0,1)  │
   ├────────────┤  └────────────┘
   │  Quick ⚠️  │
   │  O(n²)     │
   │  worst     │
   │  random    │
   │  pivot fix │
   ├────────────┤
   │  Heap      │
   │  O(n log n)│
   │  guaranteed│
   │  O(1) space│
   └────────────┘

PICK GUIDE:
  Interview: merge sort
  Linked list: merge sort
  Python: list.sort()
  k-th element: quickselect
  integer range: counting sort
```

---
*End of Sorting Algorithms Master Guide — Sean Edition*
